# Text Extraction
 Extracting text from PDF using Granite Doclings and adding image descriptions using qwen 3.

Input: PDF | 
Output: Markdown in txt format (granite_docling_output.txt)

Alternatives:
Proprietary - Gemini | 
Open Source - Gemma3, NanonetsOCR2, LightsOnOCR


In [1]:
import os
os.environ["HF_HOME"] = "D:/huggingface_cache"

In [2]:
#Import Granite

import torch
from transformers import AutoProcessor,AutoModelForImageTextToText

model_name="ibm-granite/granite-docling-258M"
device="cuda" if torch.cuda.is_available() else "cpu"

#Load Preprocessor and Model
granite_doc_processor=AutoProcessor.from_pretrained(model_name)
granite_doc_model=AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path=model_name,
    dtype=torch.bfloat16,
).to(device)

In [3]:
# Convert PDF files to page images using PyMuPDF
# Handles multiple PDFs in the documents folder

import os
import pymupdf
from PIL import Image
import io
import json

def pdf_to_images(folder_path, dpi=300, output_folder="../data/page_images"):
    """
    Converts all PDFs in folder_path to page images.
    Saves images to disk and returns sorted image_paths list.
    Also saves a metadata JSON mapping each image to its source PDF and page.
    Supports multiple PDFs — processes them in sorted filename order.
    """
    image_paths = []
    metadata    = []

    os.makedirs(output_folder, exist_ok=True)
    meta_path = os.path.join(output_folder, "metadata.json")

    pdf_files = sorted([
        f for f in os.listdir(folder_path)
        if f.lower().endswith(".pdf")
    ])

    if not pdf_files:
        print("No PDF files found in folder.")
        return [], []

    print(f"Found {len(pdf_files)} PDF(s): {pdf_files}")

    for file in pdf_files:
        pdf_path  = os.path.join(folder_path, file)
        pdf_doc   = pymupdf.open(pdf_path)
        pdf_name  = os.path.splitext(file)[0]
        num_pages = len(pdf_doc)   # store before closing

        print(f"Processing: {file} ({num_pages} pages)")

        for page_number, page in enumerate(pdf_doc):
            pix         = page.get_pixmap(dpi=dpi)
            image_bytes = pix.tobytes("png")
            image       = Image.open(io.BytesIO(image_bytes))

            image_filename = f"{pdf_name}_page_{page_number:04d}.png"
            image_path     = os.path.join(output_folder, image_filename)

            image.save(image_path)
            image_paths.append(image_path)
            metadata.append({
                "pdf_file"    : file,
                "pdf_name"    : pdf_name,
                "page_number" : page_number,
                "image_path"  : image_path,
            })

            del image

        pdf_doc.close()
        print(f"  Saved {num_pages} page images for {file}")  # use stored count

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print(f"\nTotal pages: {len(image_paths)}")
    print(f"Metadata saved to {meta_path}")
    return image_paths, metadata

In [4]:
#Generating images:
pdf_folder="../data/documents"
image_paths, metadata = pdf_to_images(pdf_folder)
print(len(image_paths), "pages loaded")

Found 1 PDF(s): ['Grand-Vitara-Petrol-Manual-latestpdf.pdf']
Processing: Grand-Vitara-Petrol-Manual-latestpdf.pdf (654 pages)
  Saved 654 page images for Grand-Vitara-Petrol-Manual-latestpdf.pdf

Total pages: 654
Metadata saved to ../data/page_images\metadata.json
654 pages loaded


In [ ]:
#Check Generated Images
import os
import json

image_folder = "../data/page_images"
meta_path    = "../data/page_images/metadata.json"

# ── Count images ─────────────────────────────────────────────────────────────
image_files = [
    f for f in os.listdir(image_folder)
    if f.endswith(".png")
]
print(f"Total page images in folder: {len(image_files)}")

# ── Load and inspect metadata ─────────────────────────────────────────────────
if os.path.exists(meta_path):
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    print(f"Total entries in metadata.json: {len(metadata)}")
    print(f"\nFirst 3 metadata entries:")
    for entry in metadata[:3]:
        print(f"  {entry}")

    print(f"\nLast 3 metadata entries:")
    for entry in metadata[-3:]:
        print(f"  {entry}")

    # Check for multiple PDFs
    pdfs_found = list(dict.fromkeys(entry["pdf_file"] for entry in metadata))
    print(f"\nPDFs found in metadata: {pdfs_found}")
    for pdf in pdfs_found:
        count = sum(1 for e in metadata if e["pdf_file"] == pdf)
        print(f"  {pdf}: {count} pages")

else:
    print("\nNo metadata.json found — metadata was not generated.")
    print("This means pdf_to_images() was run with the old code.")
    print("Re-run Cell 3 to regenerate page images with metadata.")

Total page images in folder: 654
Total entries in metadata.json: 654

First 3 metadata entries:
  {'pdf_file': 'Grand-Vitara-Petrol-Manual-latestpdf.pdf', 'pdf_name': 'Grand-Vitara-Petrol-Manual-latestpdf', 'page_number': 0, 'image_path': '../data/page_images\\Grand-Vitara-Petrol-Manual-latestpdf_page_0000.png'}
  {'pdf_file': 'Grand-Vitara-Petrol-Manual-latestpdf.pdf', 'pdf_name': 'Grand-Vitara-Petrol-Manual-latestpdf', 'page_number': 1, 'image_path': '../data/page_images\\Grand-Vitara-Petrol-Manual-latestpdf_page_0001.png'}
  {'pdf_file': 'Grand-Vitara-Petrol-Manual-latestpdf.pdf', 'pdf_name': 'Grand-Vitara-Petrol-Manual-latestpdf', 'page_number': 2, 'image_path': '../data/page_images\\Grand-Vitara-Petrol-Manual-latestpdf_page_0002.png'}

Last 3 metadata entries:
  {'pdf_file': 'Grand-Vitara-Petrol-Manual-latestpdf.pdf', 'pdf_name': 'Grand-Vitara-Petrol-Manual-latestpdf', 'page_number': 651, 'image_path': '../data/page_images\\Grand-Vitara-Petrol-Manual-latestpdf_page_0651.png'}
  {'

In [6]:
#Prompt for Granite
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."},
        ],
    },
]

prompt=granite_doc_processor.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True
)

In [ ]:
# #Running Granite for image to doctags (60s per page approx)

# batch_size=5

# import os
# import time

# progress_file = "../data/extracted_text/doctags_progress.txt"

# all_doc_tags = []

# # Load existing progress
# if os.path.exists(progress_file):

#     with open(progress_file, "r", encoding="utf-8") as f:
#         content = f.read()

#     all_doc_tags = [t for t in content.split("\n<<PAGE_BREAK>>\n") if t.strip()]

#     print(f"Resumed from {len(all_doc_tags)} processed pages")

# start_page = len(all_doc_tags)
# start_total = time.time()

# # Track pages added since last save
# pages_since_last_save = 0

# for i in range(start_page, len(image_paths),batch_size):

#     batch_start = time.time()

#     batch_paths = image_paths[i:i+batch_size]
#     batch_images = [Image.open(p) for p in batch_paths]

#     inputs=granite_doc_processor(
#         text=[prompt]*len(batch_images),
#         images=batch_images,
#         return_tensors="pt",
#         padding=True
#     ).to(device)

#     with torch.inference_mode():

#         generated_ids = granite_doc_model.generate(
#             **inputs,
#             max_new_tokens=4096,
#             use_cache=True
#         )

#     prompt_length = inputs.input_ids.shape[1]
#     trimmed_generated_ids = generated_ids[:, prompt_length:]

#     doctags = granite_doc_processor.batch_decode(
#         trimmed_generated_ids,
#         skip_special_tokens=False,
#     )

#     outputs = [o.lstrip() for o in doctags]

#     all_doc_tags.extend(outputs)
#     pages_since_last_save += len(outputs)

#     batch_time = time.time() - batch_start
#     total_time = time.time() - start_total

#     print(
#         f"Processed pages {i+1} → {i+len(batch_images)} | "
#         f"Batch time: {batch_time:.2f}s | "
#         f"Total time: {total_time/60:.2f} min"
#     )

#     # periodically clear GPU cache
#     if (i // batch_size) % 10 == 0:
#         torch.cuda.empty_cache()

#     # save every 50 NEW pages
#     if pages_since_last_save >= 50:

#         with open(progress_file, "w", encoding="utf-8") as f:
#             for tag in all_doc_tags:
#                 f.write(tag+"\n<<PAGE_BREAK>>\n")

#         print(f"Saved progress at {len(all_doc_tags)} pages")

#         pages_since_last_save = 0

#     # free memory
#     del batch_images
#     del inputs
#     del generated_ids


# # FINAL SAVE (for remaining pages <50)
# with open(progress_file, "w", encoding="utf-8") as f:
#     for tag in all_doc_tags:
#         f.write(tag+"\n<<PAGE_BREAK>>\n")

# print(f"Final save completed. Total pages saved: {len(all_doc_tags)}")

Resumed from 636 processed pages
Processed pages 637 → 641 | Batch time: 202.27s | Total time: 3.37 min
Processed pages 642 → 646 | Batch time: 195.94s | Total time: 6.64 min
Processed pages 647 → 651 | Batch time: 203.06s | Total time: 10.02 min
Processed pages 652 → 654 | Batch time: 150.25s | Total time: 12.53 min
Final save completed. Total pages saved: 654


In [11]:
# Running Granite for image to doctags (60s per batch approx)

import os
import re
import time
from PIL import Image

batch_size = 1

progress_file = "../data/extracted_text/doctags_progress.txt"
all_doc_tags  = []

# Load existing progress
if os.path.exists(progress_file):
    with open(progress_file, "r", encoding="utf-8") as f:
        content = f.read()
    all_doc_tags = [t for t in content.split("\n<<PAGE_BREAK>>\n") if t.strip()]
    print(f"Resumed from {len(all_doc_tags)} processed pages")

start_page            = len(all_doc_tags)
start_total           = time.time()
pages_since_last_save = 0

for i in range(start_page, len(image_paths), batch_size):

    batch_start  = time.time()
    batch_paths  = image_paths[i:i+batch_size]
    batch_images = [Image.open(p) for p in batch_paths]
    actual_batch = len(batch_images)

    inputs = granite_doc_processor(
        text=[prompt] * actual_batch,
        images=batch_images,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.inference_mode():
        generated_ids = granite_doc_model.generate(
            **inputs,
            max_new_tokens=8192,    # increased from 4096 to prevent truncation
            use_cache=True
        )

    prompt_length         = inputs.input_ids.shape[1]
    trimmed_generated_ids = generated_ids[:, prompt_length:]

    doctags = granite_doc_processor.batch_decode(
        trimmed_generated_ids,
        skip_special_tokens=False,
    )

    outputs = [o.lstrip() for o in doctags]

    # ── Validate each output — fix merges and detect truncations ────────────
    validated = []
    for page_offset, raw_output in enumerate(outputs):
        page_num    = i + page_offset
        close_count = raw_output.count("</doctag>")

        if close_count == 0:
            # Truncated — model ran out of tokens mid-generation
            print(f"  WARNING page {page_num+1}: truncated output (no </doctag>)")
            validated.append(raw_output)

        elif close_count == 1:
            # Perfect — exactly one complete page
            validated.append(raw_output)

        else:
            # Multiple pages merged into one string — split them apart
            print(f"  WARNING page {page_num+1}: {close_count} pages merged, splitting...")
            individual = re.findall(r'<doctag>.*?</doctag>', raw_output, re.DOTALL)
            if individual:
                validated.extend(individual)
                print(f"    → Split into {len(individual)} pages successfully")
            else:
                # Regex found nothing — keep as-is to avoid losing data
                print(f"    → Split failed, keeping as-is")
                validated.append(raw_output)

    # ── Align validated count to actual_batch ───────────────────────────────
    # Splitting may produce more outputs than images processed
    # Trim excess or pad missing to keep doctags aligned with image_paths
    if len(validated) > actual_batch:
        print(f"  NOTE: {len(validated)} outputs for {actual_batch} images — trimming to {actual_batch}")
        validated = validated[:actual_batch]
    elif len(validated) < actual_batch:
        missing = actual_batch - len(validated)
        print(f"  NOTE: {len(validated)} outputs for {actual_batch} images — padding {missing} empty entries")
        for _ in range(missing):
            validated.append("<doctag></doctag>")

    all_doc_tags.extend(validated)
    pages_since_last_save += actual_batch

    batch_time = time.time() - batch_start
    total_time = time.time() - start_total

    print(
        f"Pages {i+1}→{i+actual_batch} | "
        f"Batch: {batch_time:.1f}s | "
        f"Total: {total_time/60:.1f}min | "
        f"Processed: {len(all_doc_tags)}/{len(image_paths)}"
    )

    # Periodically clear GPU cache
    if (i // batch_size) % 10 == 0:
        torch.cuda.empty_cache()

    # Save every 10 new pages
    if pages_since_last_save >= 2:
        with open(progress_file, "w", encoding="utf-8") as f:
            for tag in all_doc_tags:
                f.write(tag + "\n<<PAGE_BREAK>>\n")
        print(f"  Saved progress at {len(all_doc_tags)} pages")
        pages_since_last_save = 0

    # Free memory
    del batch_images, inputs, generated_ids

# Final save
with open(progress_file, "w", encoding="utf-8") as f:
    for tag in all_doc_tags:
        f.write(tag + "\n<<PAGE_BREAK>>\n")

print(f"Final save completed. Total pages saved: {len(all_doc_tags)}")

Resumed from 610 processed pages
Pages 611→611 | Batch: 161.4s | Total: 2.7min | Processed: 611/654
Pages 612→612 | Batch: 161.1s | Total: 5.4min | Processed: 612/654
  Saved progress at 612 pages
Pages 613→613 | Batch: 318.4s | Total: 10.7min | Processed: 613/654
Pages 614→614 | Batch: 285.8s | Total: 15.4min | Processed: 614/654
  Saved progress at 614 pages
  WARNING page 615: truncated output (no </doctag>)
Pages 615→615 | Batch: 302.4s | Total: 20.5min | Processed: 615/654
  WARNING page 616: truncated output (no </doctag>)
Pages 616→616 | Batch: 416.8s | Total: 27.4min | Processed: 616/654
  Saved progress at 616 pages
  WARNING page 617: truncated output (no </doctag>)
Pages 617→617 | Batch: 377.5s | Total: 33.7min | Processed: 617/654
  WARNING page 618: 3 pages merged, splitting...
    → Split into 1 pages successfully
Pages 618→618 | Batch: 343.7s | Total: 39.5min | Processed: 618/654
  Saved progress at 618 pages
Pages 619→619 | Batch: 147.9s | Total: 41.9min | Processed: 61

In [13]:
#Check number of pages extracted by Granite
progress_file = "../data/extracted_text/doctags_progress.txt"

with open(progress_file, "r", encoding="utf-8") as f:
    content = f.read()

pages = [p for p in content.split("<<PAGE_BREAK>>") if p.strip()]

print("Total pages saved in doctags_progress.txt:", len(pages))

Total pages saved in doctags_progress.txt: 654


In [ ]:
# Rebuild image_paths from disk if Cell 3 was skipped
# Uses metadata.json to guarantee correct order across multiple PDFs

import os
import json

meta_path = "../data/page_images/metadata.json"

if os.path.exists(meta_path):
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    image_paths = [entry["image_path"] for entry in metadata]
    print(f"Loaded {len(image_paths)} page images from metadata.json")
else:
    # Fallback: sort by filename — works for single PDF only
    image_folder = "../data/page_images"
    image_paths  = sorted([
        os.path.join(image_folder, f)
        for f in os.listdir(image_folder)
        if f.endswith(".png") and f != "metadata.json"
    ])
    print(f"Loaded {len(image_paths)} page images from disk (no metadata.json found)")
    print("WARNING: order may be incorrect for multiple PDFs")

Loaded 654 page images from disk


In [ ]:
# from docling_core.types.doc.document import DoclingDocument, DocTagsDocument
# from PIL import Image

# progress_file = "../data/extracted_text/doctags_progress.txt"

# with open(progress_file, "r", encoding="utf-8") as f:
#     content = f.read()

# all_doc_tags = [t for t in content.split("\n<<PAGE_BREAK>>\n") if t.strip()]

# chunk_size = 50
# all_docling_docs = []
# all_docling_doc_image_ranges = []

# for i in range(0, len(all_doc_tags), chunk_size):

#     chunk_tags = all_doc_tags[i:i+chunk_size]
#     chunk_images = [Image.open(p) for p in image_paths[i:i+len(chunk_tags)]]

#     doctag_document = DocTagsDocument.from_doctags_and_image_pairs(
#         doctags=chunk_tags,
#         images=chunk_images
#     )

#     docling_document = DoclingDocument.load_from_doctags(
#         doctag_document=doctag_document,
#         document_name=f"Document_part_{i}"
#     )

#     all_docling_docs.append(docling_document)
#     all_docling_doc_image_ranges.append((i, i + len(chunk_tags)))

#     del chunk_images
#     del doctag_document

# print(f"Reconstructed {len(all_docling_docs)} document chunks")

Reconstructed 14 document chunks


In [14]:
progress_file = "../data/extracted_text/doctags_progress.txt"

with open(progress_file, "r", encoding="utf-8") as f:
    content = f.read()

pages_processed = content.count("<<PAGE_BREAK>>")

print("Pages extracted:", pages_processed)

Pages extracted: 654


In [15]:
print("Pages currently in RAM:", len(all_doc_tags))

Pages currently in RAM: 654


In [16]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

# Unload Granite if it was loaded this session
try:
    del granite_doc_model
    del granite_doc_processor
    torch.cuda.empty_cache()
    print("Granite unloaded — VRAM freed")
except NameError:
    torch.cuda.empty_cache()
    print("Granite not in session — VRAM cleared")

qwen_model_name = "Qwen/Qwen3-VL-4B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

qwen_processor = AutoProcessor.from_pretrained(qwen_model_name)

qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    qwen_model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

qwen_processor.tokenizer.padding_side = "left"
print("Qwen3-VL-4B loaded (4-bit NF4) — ready for figure description")

Granite unloaded — VRAM freed


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Qwen3-VL-4B loaded (4-bit NF4) — ready for figure description


In [29]:
import os
import json
import time
from docling_core.types.doc.document import DoclingDocument, DocTagsDocument
from PIL import Image
import torch

progress_file     = "../data/extracted_text/doctags_progress.txt"
descriptions_file = "../data/extracted_text/page_descriptions.json"

with open(progress_file, "r", encoding="utf-8") as f:
    content = f.read()
all_doc_tags = [t for t in content.split("\n<<PAGE_BREAK>>\n") if t.strip()]
print(f"Loaded {len(all_doc_tags)} doctags")
print(f"Loaded {len(image_paths)} page images")

if len(all_doc_tags) != len(image_paths):
    print(f"WARNING: doctag count ({len(all_doc_tags)}) != image count ({len(image_paths)})")
else:
    print("Counts match — safe to proceed")

if os.path.exists(descriptions_file):
    with open(descriptions_file, "r", encoding="utf-8") as f:
        page_descriptions = json.load(f)
    print(f"Resumed {len(page_descriptions)} existing page descriptions")
else:
    page_descriptions = {}
    print("Starting fresh descriptions file")

PLACEHOLDER = "<!-- image -->"

# Qwen decides count — no forced number
QWEN_PROMPT = (
    "This is a page from an industry document such as a vehicle manual. "
    "Identify every REAL figure, diagram, illustration, photograph, chart, "
    "graph, or technical drawing visible on this page.\n\n"
    "Identify visual elements corresponding to distinct image regions on the page."
    "IMPORTANT:\n"
        "- Treat EACH separate visual block, icon, or embedded image as a separate figure\n"
        "- Even if multiple visuals are part of a table, treat EACH icon or image as an individual figure\n"
        "- Do NOT group multiple visuals into one description\n"
    "IMPORTANT — Do NOT describe:\n"
    "- Text-only WARNING, NOTICE, or CAUTION boxes\n"
    "- Pure text paragraphs or lists\n"
    "- Page headers or footers\n\n"
    "For each REAL visual figure found, provide a detailed description "
    "including: what it shows, labeled components, callout numbers, "
    "and what it communicates to the reader.\n\n"
    "Return STRICTLY valid JSON only — no markdown, no code fences, "
    "no text before or after:\n"
    "{\n"
    '  "figures": [\n'
    '    {"id": 1, "description": "detailed description here"}\n'
    "  ]\n"
    "}\n\n"
    "If NO real figures exist on this page, return:\n"
    '{"figures": []}'
)


def resize_image(img, max_size=1024):
    img = img.copy()
    img.thumbnail((max_size, max_size))
    return img


def describe_page_with_qwen(page_img, max_tokens=768):
    torch.cuda.empty_cache()
    page_img = resize_image(page_img)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": page_img},
            {"type": "text",  "text": QWEN_PROMPT},
        ],
    }]

    text_input = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

    inputs = qwen_processor(
        text=[text_input],
        images=[page_img],
        return_tensors="pt",
        padding=True,
    ).to(qwen_model.device)

    with torch.inference_mode():
        generated_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
        )

    trimmed    = generated_ids[0][len(inputs.input_ids[0]):]
    raw_output = qwen_processor.decode(
        trimmed, skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()

    del inputs, generated_ids
    torch.cuda.empty_cache()
    return raw_output


def parse_qwen_output(raw_output):
    """
    Parse Qwen JSON output.
    Returns (figures_list, parse_success).
    figures_list is a list of description strings.
    """
    cleaned = raw_output.strip()

    # Strip markdown fences if model added them
    if cleaned.startswith("```"):
        lines   = cleaned.split("\n")
        cleaned = "\n".join(lines[1:])
    if cleaned.endswith("```"):
        lines   = cleaned.split("\n")
        cleaned = "\n".join(lines[:-1])
    cleaned = cleaned.strip()

    try:
        parsed  = json.loads(cleaned)
        figures = [
            f["description"]
            for f in parsed.get("figures", [])
            if isinstance(f.get("description"), str)
            and len(f["description"]) > 20   # skip empty/junk entries
        ]
        return figures, True

    except Exception as e:
        print(f"  JSON parse failed: {e}")
        # If raw output has real content, store it as one description
        if len(raw_output) > 50:
            return [raw_output], False
        return [], False


# ── Main loop ────────────────────────────────────────────────────────────────
stats = {
    "figures"        : 0,
    "no_figures"     : 0,
    "skipped"        : 0,
    "parse_failures" : 0,
    "retries"        : 0,
    "errors"         : 0,
}

for page_idx in range(len(all_doc_tags)):

    page_key = f"page_{page_idx}"

    if page_key in page_descriptions:
        stats["skipped"] += 1
        if page_idx % 50 == 0:
            print(f"Page {page_idx+1:>4}: skipping (already done)")
        continue

    try:
        page_image  = Image.open(image_paths[page_idx])
        doctag_doc  = DocTagsDocument.from_doctags_and_image_pairs(
            doctags=[all_doc_tags[page_idx]], images=[page_image]
        )
        docling_doc = DoclingDocument.load_from_doctags(
            doctag_document=doctag_doc, document_name=page_key
        )
        page_md           = docling_doc.export_to_markdown()
        placeholder_count = page_md.count(PLACEHOLDER)
        del doctag_doc, docling_doc, page_md

        if placeholder_count == 0:
            page_descriptions[page_key] = {
                "placeholder_count" : 0,
                "figures"           : [],
                "source_image"      : image_paths[page_idx],
            }
            stats["no_figures"] += 1
            if stats["no_figures"] % 20 == 0:
                with open(descriptions_file, "w", encoding="utf-8") as f:
                    json.dump(page_descriptions, f, indent=2)
            page_image.close()
            continue

        print(f"Page {page_idx+1:>4}: {placeholder_count} placeholder(s) → Qwen...")
        t0 = time.time()

        raw_output = describe_page_with_qwen(page_image, max_tokens=768)
        figures, parse_ok = parse_qwen_output(raw_output)

        # Retry once if parse failed
        if not parse_ok:
            print(f"  Parse failed — retrying...")
            stats["retries"] += 1
            raw_output        = describe_page_with_qwen(page_image, max_tokens=768)
            figures, parse_ok = parse_qwen_output(raw_output)
            if not parse_ok:
                stats["parse_failures"] += 1

        page_image.close()

        # Log mismatch between Granite placeholders and Qwen figures
        # This is informational only — we do not force alignment here
        qwen_count = len(figures)
        if qwen_count != placeholder_count:
            print(
                f"  NOTE: Granite={placeholder_count} placeholders, "
                f"Qwen found={qwen_count} figures"
            )

        page_descriptions[page_key] = {
            "placeholder_count" : placeholder_count,
            "qwen_figure_count" : qwen_count,
            "figures"           : figures,
            "source_image"      : image_paths[page_idx],
        }

        elapsed = time.time() - t0
        print(
            f"       → {qwen_count} figure(s) | "
            f"parse={'✓' if parse_ok else '✗'} | {elapsed:.1f}s"
        )
        stats["figures"] += 1

    except Exception as e:
        print(f"Page {page_idx+1:>4}: ERROR — {e}")
        page_descriptions[page_key] = {
            "placeholder_count" : 0,
            "qwen_figure_count" : 0,
            "figures"           : [],
            "source_image"      : image_paths[page_idx],
        }
        stats["errors"] += 1

    with open(descriptions_file, "w", encoding="utf-8") as f:
        json.dump(page_descriptions, f, indent=2)

# Final save
with open(descriptions_file, "w", encoding="utf-8") as f:
    json.dump(page_descriptions, f, indent=2)

print(f"""
Done.
  Pages with figures : {stats['figures']}
  Pages no figures   : {stats['no_figures']}
  Skipped (resumed)  : {stats['skipped']}
  Parse failures     : {stats['parse_failures']}
  Retries            : {stats['retries']}
  Errors             : {stats['errors']}
  Total saved        : {len(page_descriptions)}
""")

Loaded 654 doctags
Loaded 654 page images
Counts match — safe to proceed
Resumed 494 existing page descriptions
Page    1: skipping (already done)
Page   51: skipping (already done)
Page  101: skipping (already done)
Page  151: skipping (already done)
Page  201: skipping (already done)
Page  251: skipping (already done)
Page  301: skipping (already done)
Page  351: skipping (already done)
Page  401: skipping (already done)
Page  451: skipping (already done)
Page  495: 2 placeholder(s) → Qwen...
       → 2 figure(s) | parse=✓ | 25.4s
Page  496: 1 placeholder(s) → Qwen...
       → 1 figure(s) | parse=✓ | 11.3s
Page  497: 1 placeholder(s) → Qwen...
       → 1 figure(s) | parse=✓ | 11.2s
Page  498: 1 placeholder(s) → Qwen...
       → 1 figure(s) | parse=✓ | 14.1s
Page  499: 2 placeholder(s) → Qwen...
  NOTE: Granite=2 placeholders, Qwen found=4 figures
       → 4 figure(s) | parse=✓ | 37.8s
Page  500: 3 placeholder(s) → Qwen...
       → 3 figure(s) | parse=✓ | 19.9s
Page  501: 2 placeholde

In [32]:
import json
import os
from docling_core.types.doc.document import DoclingDocument, DocTagsDocument
from PIL import Image

progress_file     = "../data/extracted_text/doctags_progress.txt"
descriptions_file = "../data/extracted_text/page_descriptions.json"
output_path       = "../data/extracted_text/enriched_output.txt"

with open(progress_file, "r", encoding="utf-8") as f:
    content = f.read()
all_doc_tags = [t for t in content.split("\n<<PAGE_BREAK>>\n") if t.strip()]

with open(descriptions_file, "r", encoding="utf-8") as f:
    page_descriptions = json.load(f)

print(f"Loaded {len(all_doc_tags)} doctags")
print(f"Loaded {len(page_descriptions)} page descriptions")

PLACEHOLDER  = "<!-- image -->"
chunk_size   = 50
total_chunks = (len(all_doc_tags) + chunk_size - 1) // chunk_size
total_placeholders_injected = 0

os.makedirs(os.path.dirname(output_path), exist_ok=True)


def merge_similar_figures(figures, target_count):
    """
    Merge figures list down to target_count by combining adjacent entries.
    Combines the shortest adjacent pair first — most likely to be related.
    """
    result = list(figures)
    while len(result) > target_count:
        min_len = float("inf")
        min_idx = 0
        for j in range(len(result) - 1):
            combined_len = len(result[j]) + len(result[j+1])
            if combined_len < min_len:
                min_len = combined_len
                min_idx = j
        merged = result[min_idx] + " " + result[min_idx+1]
        result = result[:min_idx] + [merged] + result[min_idx+2:]
    return result


with open(output_path, "w", encoding="utf-8") as out_file:

    for chunk_idx in range(0, len(all_doc_tags), chunk_size):

        chunk_num  = chunk_idx // chunk_size
        chunk_tags = all_doc_tags[chunk_idx:chunk_idx+chunk_size]
        chunk_len  = len(chunk_tags)

        print(f"Exporting chunk {chunk_num+1}/{total_chunks}...")

        per_page_md     = []
        injection_queue = []

        for i in range(chunk_len):
            global_page_idx = chunk_idx + i
            page_key        = f"page_{global_page_idx}"
            page_image      = Image.open(image_paths[global_page_idx])

            pg_doctag_doc  = DocTagsDocument.from_doctags_and_image_pairs(
                doctags=[chunk_tags[i]],
                images=[page_image]
            )
            pg_docling_doc = DoclingDocument.load_from_doctags(
                doctag_document=pg_doctag_doc,
                document_name=page_key
            )
            pg_md = pg_docling_doc.export_to_markdown()
            per_page_md.append(pg_md)

            page_image.close()
            del pg_doctag_doc, pg_docling_doc

            # ── Build injection queue ─────────────────────────────────────────
            ph_count  = pg_md.count(PLACEHOLDER)
            page_data = page_descriptions.get(page_key, {})
            figures   = page_data.get("figures", [])

            if ph_count == 0:
                pass

            elif len(figures) == 0:
                for _ in range(ph_count):
                    injection_queue.append(None)

            elif len(figures) > ph_count:
                merged = merge_similar_figures(figures, ph_count)
                for fig_text in merged:
                    injection_queue.append(fig_text)

            elif len(figures) < ph_count:
                for fig_text in figures:
                    injection_queue.append(fig_text)
                for _ in range(ph_count - len(figures)):
                    injection_queue.append(None)

            else:
                for fig_text in figures:
                    injection_queue.append(fig_text)

        # Combine per-page markdowns
        chunk_markdown = "\n\n".join(per_page_md)
        del per_page_md

        # ── Inject descriptions ───────────────────────────────────────────────
        # Replace each <!-- image --> line with a clean image description
        # paragraph. The placeholder itself is removed — it adds no value
        # to the embedding model and clutters the markdown.
        lines        = chunk_markdown.split("\n")
        result_lines = []
        ph_cursor    = 0

        for line in lines:
            if PLACEHOLDER not in line:
                result_lines.append(line)
                continue

            # Replace placeholder line entirely — do not keep it
            total_placeholders_injected += 1

            if ph_cursor < len(injection_queue):
                text = injection_queue[ph_cursor]
                ph_cursor += 1
            else:
                text = None

            if text:
                # Clean standalone paragraph — optimal for embedding
                result_lines.append(f"**[Image]:** {text}")
            # If no description — omit entirely, adds no value to embeddings

            result_lines.append("")  # blank line after for paragraph separation

        out_file.write("\n".join(result_lines))
        out_file.write("\n\n")

        del chunk_markdown, result_lines, injection_queue
        print(f"  Chunk {chunk_num+1} written")

print(f"""
Done.
  Output saved to   : {output_path}
  Total placeholders: {total_placeholders_injected}
""")

Loaded 654 doctags
Loaded 654 page descriptions
Exporting chunk 1/14...
  Chunk 1 written
Exporting chunk 2/14...
  Chunk 2 written
Exporting chunk 3/14...
  Chunk 3 written
Exporting chunk 4/14...
  Chunk 4 written
Exporting chunk 5/14...
  Chunk 5 written
Exporting chunk 6/14...
  Chunk 6 written
Exporting chunk 7/14...
  Chunk 7 written
Exporting chunk 8/14...
  Chunk 8 written
Exporting chunk 9/14...
  Chunk 9 written
Exporting chunk 10/14...
  Chunk 10 written
Exporting chunk 11/14...
  Chunk 11 written
Exporting chunk 12/14...
  Chunk 12 written
Exporting chunk 13/14...
  Chunk 13 written
Exporting chunk 14/14...
  Chunk 14 written

Done.
  Output saved to   : ../data/extracted_text/enriched_output.txt
  Total placeholders: 1255



In [33]:
#Offload QWEN

import torch
import gc

try:
    del qwen_model
    del qwen_processor
    torch.cuda.empty_cache()
    gc.collect()
    print("Qwen offloaded — VRAM freed")
except NameError:
    print("Qwen not in session")

Qwen offloaded — VRAM freed
